In [1]:
import sys
import os

sys.path.append(os.path.abspath("./src"))

In [2]:
import sys
print(sys.executable)
print(sys.version)

/opt/conda/envs/python3/bin/python
3.11.12 | packaged by conda-forge | (main, Apr 10 2025, 22:23:25) [GCC 13.3.0]


In [3]:
!{sys.executable} -m pip install pandas

  Using cached pandas-3.0.1-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.1-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)


In [4]:
!{sys.executable} -m pip install numpy networkx matplotlib

In [5]:
!{sys.executable} -m pip install pyarrow fastparquet

  Using cached pyarrow-23.0.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached fastparquet-2025.12.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (4.4 kB)
  Using cached cramjam-2.11.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
Using cached pyarrow-23.0.1-cp311-cp311-manylinux_2_28_x86_64.whl (47.6 MB)
Using cached fastparquet-2025.12.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.8 MB)
Using cached cramjam-2.11.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.0 MB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastparquet] [cramjam]


In [1]:
# ─────────────────────────────────────────────
# Imports (Rigetti Workspace)
# ─────────────────────────────────────────────

import sys
sys.path.append("./src")

import json
from pathlib import Path
import networkx as nx
import pandas as pd
import numpy as np

from pyquil import Program
from pyquil.api import get_qc

from qubo_builder import build_ising, QUBOParams
from qaoa_model import build_qaoa_circuit
from qubo_builder import ising_to_qaoa_coefficients

print("Imports complete.")

Imports complete.


In [2]:
import graph_utils
print("src import working")

src import working


In [3]:
# ─────────────────────────────────────────────
# Load Layer 2 Results
# ─────────────────────────────────────────────

LOAD_PATH = Path("../results/problemB_qaoa_layer2/layer2_results.json")

with open(LOAD_PATH, "r") as f:
    layer2 = json.load(f)

clusters = [set(c) for c in layer2["clusters"]]
selected_indices = layer2["selected_indices"]
cluster_results = layer2["cluster_results"]

print("Loaded frozen Layer 2 results.")
print("Selected hardware clusters:", selected_indices)

Loaded frozen Layer 2 results.
Selected hardware clusters: [11, 6, 5]


In [4]:
# ─────────────────────────────────────────────
# Load Problem B Graph
# ─────────────────────────────────────────────

dfB = pd.read_parquet('../data/problemB.parquet')

def df_to_graph(df):
    G = nx.Graph()
    for _, r in df.iterrows():
        G.add_edge(int(r.node_1), int(r.node_2), weight=float(r.weight))
    return G

GB = df_to_graph(dfB)

print("Problem B loaded:", GB.number_of_nodes(), "nodes")

Problem B loaded: 180 nodes


In [5]:
# ─────────────────────────────────────────────
# Connect to Ankaa-3 (for compilation only)
# ─────────────────────────────────────────────

qc = get_qc("Ankaa-3", as_qvm=False)

print("Connected to:", qc.name)

Connected to: Ankaa-3


In [6]:
# ─────────────────────────────────────────────
# Compile Audit (Correct QCS Native Path)
# ─────────────────────────────────────────────

from pyquil import Program
from pyquil.gates import H, RX, RZ, ISWAP
from pyquil.api import get_qc

qc = get_qc("Ankaa-3", as_qvm=False)

audit_results = []

for idx in selected_indices:

    print(f"\n=== Cluster {idx} ===")

    r = cluster_results[idx]
    cluster_nodes = clusters[idx]
    subG = GB.subgraph(cluster_nodes).copy()

    gamma = r["gamma"][0] if isinstance(r["gamma"], list) else r["gamma"]
    beta = r["beta"][0] if isinstance(r["beta"], list) else r["beta"]

    nodes_list = list(subG.nodes())
    node_to_qubit = {n: i for i, n in enumerate(nodes_list)}
    n_qubits = len(nodes_list)

    prog = Program()

    # Initial layer
    for q in range(n_qubits):
        prog += H(q)

    # Cost layer
    for (u, v, data) in subG.edges(data=True):
        q1 = node_to_qubit[u]
        q2 = node_to_qubit[v]
        w = data["weight"]

        prog += ISWAP(q1, q2)
        prog += RZ(2 * gamma * w, q1)
        prog += RZ(2 * gamma * w, q2)
        prog += ISWAP(q1, q2)

    # Mixer
    for q in range(n_qubits):
        prog += RX(2 * beta, q)

    # ───────── Compile to native without job submission ─────────
    native_prog = qc.compiler.quil_to_native_quil(prog)

    total_2q = 0
    iswap_count = 0

    for inst in native_prog:
        if hasattr(inst, "qubits") and len(inst.qubits) == 2:
            total_2q += 1
            if inst.name.upper() == "ISWAP":
                iswap_count += 1

    expected_iswap = subG.number_of_edges() * 2
    overhead_factor = iswap_count / expected_iswap if expected_iswap > 0 else 0

    print("Edges:", subG.number_of_edges())
    print("Expected iSWAP:", expected_iswap)
    print("Actual iSWAP:", iswap_count)
    print("Total 2Q gates:", total_2q)
    print("Routing overhead factor:", round(overhead_factor, 3))

    audit_results.append({
        "cluster_id": idx,
        "edges": subG.number_of_edges(),
        "expected_iSWAP": expected_iswap,
        "actual_iSWAP": iswap_count,
        "total_2Q": total_2q,
        "overhead_factor": overhead_factor
    })

import pandas as pd
audit_df = pd.DataFrame(audit_results)
display(audit_df)


=== Cluster 11 ===
Edges: 10
Expected iSWAP: 20
Actual iSWAP: 9
Total 2Q gates: 9
Routing overhead factor: 0.45

=== Cluster 6 ===
Edges: 9
Expected iSWAP: 18
Actual iSWAP: 12
Total 2Q gates: 12
Routing overhead factor: 0.667

=== Cluster 5 ===
Edges: 9
Expected iSWAP: 18
Actual iSWAP: 60
Total 2Q gates: 60
Routing overhead factor: 3.333


,cluster_id,edges,expected_iSWAP,actual_iSWAP,total_2Q,overhead_factor
0,11,10,20,9,9,0.450000
1,6,9,18,12,12,0.666667
2,5,9,18,60,60,3.333333


In [7]:
print(qc.qubit_topology())

Graph with 82 nodes and 140 edges


In [8]:
from pyquil.api import get_qc
import inspect

qc = get_qc("Ankaa-3", as_qvm=False)

print("QC type:", type(qc))
print("Compiler type:", type(qc.compiler))
print("Quantum processor type:", type(qc.compiler.quantum_processor))

print("\nAvailable attributes of quantum_processor:")
print(dir(qc.compiler.quantum_processor))

QC type: <class 'pyquil.api._quantum_computer.QuantumComputer'>
Compiler type: <class 'pyquil.api._compiler.QPUCompiler'>
Quantum processor type: <class 'pyquil.quantum_processor.qcs.QCSQuantumProcessor'>

Available attributes of quantum_processor:
['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_isa', 'noise_model', 'quantum_processor_id', 'qubit_topology', 'qubits', 'to_compiler_isa']


In [9]:

import networkx as nx

qp = qc.compiler.quantum_processor

topo = qp.qubit_topology()  # <-- this is the key

print("Topology type:", type(topo))

# Convert to NetworkX graph
G_phys = nx.Graph()
G_phys.add_edges_from(topo.edges)

print("Physical qubits:", G_phys.number_of_nodes())
print("Couplers:", G_phys.number_of_edges())

print("Is connected:", nx.is_connected(G_phys))

Topology type: <class 'networkx.classes.graph.Graph'>
Physical qubits: 82
Couplers: 140
Is connected: True


In [10]:
# ─────────────────────────────────────────────
# Find connected 12-qubit blocks
# ─────────────────────────────────────────────

import random

def find_connected_block(G, size, trials=500):
    nodes = list(G.nodes())
    for _ in range(trials):
        start = random.choice(nodes)
        visited = {start}
        frontier = [start]

        while frontier and len(visited) < size:
            cur = frontier.pop()
            for nbr in G.neighbors(cur):
                if nbr not in visited:
                    visited.add(nbr)
                    frontier.append(nbr)
                    if len(visited) >= size:
                        break

        if len(visited) == size:
            return list(visited)

    return None

blocks12 = []
for _ in range(3):
    block = find_connected_block(G_phys, 12)
    assert block is not None
    blocks12.append(block)

print("Connected 12-qubit blocks:")
for b in blocks12:
    print(sorted(b))

Connected 12-qubit blocks:
[47, 53, 54, 55, 60, 61, 62, 67, 68, 69, 74, 75]
[50, 56, 57, 58, 63, 64, 65, 70, 71, 72, 77, 78]
[57, 63, 64, 65, 70, 71, 72, 73, 77, 78, 79, 80]


## Embedding Refinement Phase (Cluster 5)

Cluster 5 exhibited excessive routing overhead (60 native 2Q gates) under naive logical-to-physical mapping.

We now perform a controlled embedding refinement:

1. Logical nodes are sorted by internal degree (descending).
2. Multiple randomized qubit orderings are evaluated.
3. The compilation with minimum native 2Q count is selected.

This ensures routing overhead is minimized before hardware execution.

In [11]:
# ─────────────────────────────────────────────
# Embedding Refinement for Cluster 5
# ─────────────────────────────────────────────

import random

idx = 5  # problematic cluster
r = cluster_results[idx]
cluster_nodes = clusters[idx]
subG = GB.subgraph(cluster_nodes).copy()

gamma = r["gamma"][0] if isinstance(r["gamma"], list) else r["gamma"]
beta = r["beta"][0] if isinstance(r["beta"], list) else r["beta"]

best_2q = 1e9
best_mapping = None
best_native_prog = None

logical_nodes = list(subG.nodes())

# Sort by internal degree (descending)
logical_nodes_sorted = sorted(
    logical_nodes,
    key=lambda n: subG.degree(n),
    reverse=True
)

TRIALS = 40  # increase to 50 if time permits

for trial in range(TRIALS):

    # Slightly randomize after degree sort
    nodes_list = logical_nodes_sorted.copy()
    random.shuffle(nodes_list)

    node_to_qubit = {nodes_list[i]: i for i in range(len(nodes_list))}

    prog = Program()

    # Initial layer
    for logical in nodes_list:
        prog += H(node_to_qubit[logical])

    # Cost layer
    for (u, v, data) in subG.edges(data=True):
        q1 = node_to_qubit[u]
        q2 = node_to_qubit[v]
        w = data["weight"]

        prog += ISWAP(q1, q2)
        prog += RZ(2 * gamma * w, q1)
        prog += RZ(2 * gamma * w, q2)
        prog += ISWAP(q1, q2)

    # Mixer
    for logical in nodes_list:
        prog += RX(2 * beta, node_to_qubit[logical])

    native_prog = qc.compiler.quil_to_native_quil(prog)

    total_2q = 0
    for inst in native_prog:
        if hasattr(inst, "qubits") and len(inst.qubits) == 2:
            total_2q += 1

    if total_2q < best_2q:
        best_2q = total_2q
        best_mapping = node_to_qubit.copy()
        best_native_prog = native_prog

print("Best compiled 2Q after refinement:", best_2q)

Best compiled 2Q after refinement: 30


## Final Hardware-Executable Cluster Selection

After topology-aware embedding refinement, the compiled native two-qubit gate counts are:

- Cluster 11 → 9 native 2Q gates
- Cluster 6  → 12 native 2Q gates
- Cluster 5  → 30 native 2Q gates (after embedding refinement)

This produces a controlled hardware complexity gradient suitable for envelope validation.

In [12]:
# ─────────────────────────────────────────────
# Final Hardware Survival Estimates
# ─────────────────────────────────────────────

F = 0.9852366338446036
T2 = 22.07562115774664
t_iswap = 0.2

compiled_2q = [9, 12, 30]
cluster_ids = [11, 6, 5]

results = []

for cid, N2Q in zip(cluster_ids, compiled_2q):
    
    survival = (F ** N2Q) * np.exp(-(N2Q * t_iswap) / T2)
    
    results.append({
        "cluster_id": cid,
        "compiled_2Q": N2Q,
        "predicted_survival": survival
    })

pd.DataFrame(results)

,cluster_id,compiled_2Q,predicted_survival
0,11,9,0.806220
1,6,12,0.750363
2,5,30,0.487729
